In [1]:
import numpy as np
import re
import keras_nlp
import keras
import random
import tensorflow as tf

In [2]:
def readFasta(inpFile):
    _seq = []
    _id = []
    for line in open(inpFile):
        if line.startswith('>'):
           _id.append(line.replace('>','').split()[0])
        else:
           _seq.append(line.replace('\n',''))
    return _seq, _id

In [3]:
[train_seq,train_id] = readFasta('C:/Users/dell/Desktop/autobioseqpy2.0_20200805/examples/m5U/data/Full_train_positive.txt')
[test_seq,test_id] = readFasta('C:/Users/dell/Desktop/autobioseqpy2.0_20200805/examples/m5U/data/Full_test_positive.txt')
index = 20
train_seq = [s[:index] + 'M' + s[index + 1:] for s in train_seq]
test_seq = [s[:index] + 'M' + s[index + 1:] for s in test_seq]
print(train_seq)

['...']


In [4]:
train_seq_split = []
for v in range(len(train_seq)):
    result = ' '.join([train_seq[v][i:i+3] for i in range(0, len(train_seq[v])-2, 1)])
    train_seq_split = np.append(train_seq_split,result)

test_seq_split = []
for v in range(len(test_seq)):
    result = ' '.join([test_seq[v][i:i+3] for i in range(0, len(test_seq[v])-2, 1)])
    test_seq_split = np.append(test_seq_split,result)

print(train_seq_split)

['AGC GCC CCC CCU CUG UGC GCC CCC CCG CGG GGG GGA GAA AAU AUC UCG CGG GGA GAM AMG MGG GGA GAG AGG GGC GCU CUG UGC GCU CUC UCU CUC UCA CAG AGC GCU CUC UCC CCU'
 'ACA CAA AAC ACC CCU CUU UUU UUG UGA GAA AAU AUU UUG UGC GCU CUC UCU CUA UAM AMU MUC UCG CGU GUC UCA CAC ACU CUA UAG AGA GAA AAG AGU GUC UCC CCA CAA AAU AUG'
 'GUC UCA CAG AGG GGG GGG GGC GCA CAC ACC CCG CGC GCU CUG UGG GGG GGA GAU AUM UMA MAG AGA GAA AAA AAC ACC CCU CUG UGG GGU GUC UCC CCC CCU CUU UUG UGA GAC ACA'
 ...
 'ACC CCC CCU CUG UGC GCG CGC GCC CCU CUU UUU UUG UGU GUG UGA GAC ACA CAU AUM UMC MCC CCG CGC GCA CAG AGC GCG CGC GCU CUG UGC GCA CAG AGA GAG AGA GAG AGG GGG'
 'UUC UCA CAU AUC UCU CUC UCG CGA GAA AAC ACC CCC CCC CCU CUC UCG CGA GAA AAM AMC MCA CAC ACU CUG UGU GUU UUU UUC UCC CCC CCA CAU AUA UAC ACU CUA UAA AAC ACU'
 'AGG GGC GCU CUC UCC CCA CAG AGG GGU GUG UGA GAG AGG GGA GAU AUC UCC CCA CAM AMU MUU UUC UCA CAC ACU CUG UGU GUC UCU CUU UUU UUU UUC UCC CCA CAG AGC GCA CAU']


In [15]:
vocab = ['[UNK]', '[CLS]', '[SEP]', '[PAD]','[MASK]']
nucleotide = ['A', 'U', 'C', 'G']

combinations = []
for i in nucleotide:
    for j in nucleotide:
        for k in nucleotide:
            combination = i + j + k
            combinations.append(combination)
            
combinations_1 = []
combinations_2 = []
combinations_3 = []

for n1 in ['M']:
    for n2 in nucleotide:
        for n3 in nucleotide:
            combination = n1 + n2 + n3
            combinations_1.append(combination)

for n1 in nucleotide:
    for n2 in ['M']:
        for n3 in nucleotide:
            combination = n1 + n2 + n3
            combinations_2.append(combination)

for n1 in nucleotide:
    for n2 in nucleotide:
        combination = n1 + n2 + 'M'
        combinations_3.append(combination)

word = np.concatenate((vocab,combinations,combinations_1,combinations_2,combinations_3))
print(word)

['[UNK]' '[CLS]' '[SEP]' '[PAD]' '[MASK]' 'AAA' 'AAU' 'AAC' 'AAG' 'AUA'
 'AUU' 'AUC' 'AUG' 'ACA' 'ACU' 'ACC' 'ACG' 'AGA' 'AGU' 'AGC' 'AGG' 'UAA'
 'UAU' 'UAC' 'UAG' 'UUA' 'UUU' 'UUC' 'UUG' 'UCA' 'UCU' 'UCC' 'UCG' 'UGA'
 'UGU' 'UGC' 'UGG' 'CAA' 'CAU' 'CAC' 'CAG' 'CUA' 'CUU' 'CUC' 'CUG' 'CCA'
 'CCU' 'CCC' 'CCG' 'CGA' 'CGU' 'CGC' 'CGG' 'GAA' 'GAU' 'GAC' 'GAG' 'GUA'
 'GUU' 'GUC' 'GUG' 'GCA' 'GCU' 'GCC' 'GCG' 'GGA' 'GGU' 'GGC' 'GGG' 'MAA'
 'MAU' 'MAC' 'MAG' 'MUA' 'MUU' 'MUC' 'MUG' 'MCA' 'MCU' 'MCC' 'MCG' 'MGA'
 'MGU' 'MGC' 'MGG' 'AMA' 'AMU' 'AMC' 'AMG' 'UMA' 'UMU' 'UMC' 'UMG' 'CMA'
 'CMU' 'CMC' 'CMG' 'GMA' 'GMU' 'GMC' 'GMG' 'AAM' 'AUM' 'ACM' 'AGM' 'UAM'
 'UUM' 'UCM' 'UGM' 'CAM' 'CUM' 'CCM' 'CGM' 'GAM' 'GUM' 'GCM' 'GGM']


In [21]:
tokenizer = keras_nlp.models.BertTokenizer(word, lowercase=False)

In [22]:
input_train = tokenizer(train_seq_split).to_tensor()

In [23]:
input_train[0]

<tf.Tensor: shape=(39,), dtype=int32, numpy=
array([ 19,  63,  47,  46,  44,  35,  63,  47,  48,  52,  68,  65,  53,
         6,  11,  32,  52,  65, 113,  88,  84,  65,  56,  20,  67,  62,
        44,  35,  62,  43,  30,  43,  29,  40,  19,  62,  43,  31,  46])>

In [24]:
Preprocessor=keras_nlp.models.BertMaskedLMPreprocessor(
    tokenizer,
    sequence_length=39,
    mask_selection_rate=0.15,
    mask_selection_length=10,
    mask_token_rate=0.8,
    random_token_rate=0.1,
)

In [25]:
input=Preprocessor(train_seq_split)
input[0]['token_ids'][0]

<tf.Tensor: shape=(39,), dtype=int32, numpy=
array([  1,  19,  63,  47,  46,   4,  35,  63,  47,  48,  52,  68,  65,
        53,   6,  11,  32,  52,  65, 113,   4,  84,  65,  56,  20,  67,
        62,   4,  35,  62,  43,  30,  43,  29,  40,  19,   4,  43,   2])>

In [26]:
masked_lm = keras_nlp.models.BertMaskedLM.from_preset(
    'bert_tiny_en_uncased',
    preprocessor=None,
)

In [27]:
masked_lm.fit(x=input[0], y=input[1], epochs=50)

Epoch 1/50
93/93 [==============================] - 25s 119ms/step - loss: 7.0525 - sparse_categorical_accuracy: 0.3023
Epoch 2/50
93/93 [==============================] - 3s 27ms/step - loss: 6.0100 - sparse_categorical_accuracy: 0.4000
Epoch 3/50
93/93 [==============================] - 2s 26ms/step - loss: 5.1534 - sparse_categorical_accuracy: 0.4000
Epoch 4/50
93/93 [==============================] - 2s 26ms/step - loss: 4.4391 - sparse_categorical_accuracy: 0.4000
Epoch 5/50
93/93 [==============================] - 2s 26ms/step - loss: 3.9159 - sparse_categorical_accuracy: 0.4000
Epoch 6/50
93/93 [==============================] - 2s 26ms/step - loss: 3.5375 - sparse_categorical_accuracy: 0.4000
Epoch 7/50
93/93 [==============================] - 2s 26ms/step - loss: 3.2594 - sparse_categorical_accuracy: 0.4000
Epoch 8/50
93/93 [==============================] - 2s 26ms/step - loss: 3.0618 - sparse_categorical_accuracy: 0.4000
Epoch 9/50
93/93 [==============================] - 2s

In [75]:
test_seq = []
label = []
for i in range(len(test_seq_split)):
    new = test_seq_split[i].split()
    Modification = [new[18],new[19],new[20]]
    new[18] = 'MASK'
    new[19] = 'MASK'
    new[20] = 'MASK'
    result = ' '.join(new)
    test_seq = np.append(test_seq,result)
    label = np.append(label,Modification)
label=label.reshape(len(test_seq_split),3)
label

array([['CAM', 'AMA', 'MAU'],
       ['UUM', 'UMU', 'MUU'],
       ['GAM', 'AMU', 'MUC'],
       ...,
       ['GUM', 'UMG', 'MGU'],
       ['GCM', 'CMU', 'MUC'],
       ['ACM', 'CMC', 'MCA']], dtype='<U32')

In [76]:
test_seq

array(['GUU UUA UAU AUA UAA AAU AUU UUA UAU AUA UAA AAG AGA GAA AAC ACA CAC ACA MASK MASK MASK AUC UCC CCC CCC CCA CAU AUU UUC UCU CUG UGA GAC ACA CAC ACA CAG AGA GAG',
       'UCC CCU CUA UAU AUA UAA AAG AGA GAU AUC UCA CAA AAG AGG GGU GUC UCU CUU MASK MASK MASK UUC UCC CCA CAU AUC UCC CCC CCU CUG UGG GGC GCA CAU AUU UUU UUC UCC CCA',
       'ACA CAU AUA UAU AUU UUG UGU GUC UCA CAG AGG GGA GAG AGU GUG UGG GGG GGA MASK MASK MASK UCG CGA GAA AAC ACC CCC CCA CAC ACG CGC GCC CCU CUC UCC CCA CAG AGG GGG',
       'GUU UUA UAA AAC ACC CCG CGA GAA AAA AAG AGG GGU GUU UUG UGG GGU GUG UGG MASK MASK MASK UCG CGA GAU AUC UCC CCC CCA CAC ACC CCC CCA CAG AGG GGG GGA GAC ACG CGG',
       'UUU UUA UAA AAU AUC UCU CUG UGA GAG AGG GGG GGU GUC UCC CCA CAG AGG GGG MASK MASK MASK UCA CAA AAG AGU GUC UCC CCC CCU CUG UGU GUU UUC UCG CGG GGG GGC GCG CGU',
       'GAA AAG AGG GGU GUG UGG GGC GCC CCA CAG AGC GCA CAC ACG CGG GGG GGG GGA MASK MASK MASK GCG CGC GCU CUG UGC GCG CGG GGC GCA CAA AAA AAC ACC CCU CUA 

In [77]:
test_input=tokenizer(test_seq).to_tensor()

In [78]:
test_input[0]

<tf.Tensor: shape=(39,), dtype=int32, numpy=
array([58, 25, 22,  9, 21,  6, 10, 25, 22,  9, 21,  8, 17, 53,  7, 13, 39,
       13,  0,  0,  0, 11, 31, 47, 47, 45, 38, 10, 27, 30, 44, 33, 55, 13,
       39, 13, 40, 17, 56])>

In [88]:
padding_mask = tf.ones_like(test_input)
segment_ids = tf.zeros_like(test_input)
zero_indices = tf.where(tf.equal(test_input, 0))
reshaped_tensor = tf.reshape(zero_indices[:, 1:], (738, 3))

<tf.Tensor: shape=(738, 3), dtype=int64, numpy=
array([[18, 19, 20],
       [18, 19, 20],
       [18, 19, 20],
       ...,
       [18, 19, 20],
       [18, 19, 20],
       [18, 19, 20]], dtype=int64)>

In [89]:
test = {'token_ids': test_input,
           'padding_mask':padding_mask,
           'segment_ids':segment_ids,
           'mask_positions':reshaped_tensor}

In [90]:
prediction=masked_lm.predict(test)

24/24 [==============================] - 4s 54ms/step


In [91]:
prediction

array([[[ 2.9288788 ,  0.2751841 , -0.25666562, ..., -2.4164436 ,
         -2.5495758 , -5.3214736 ],
        [ 3.1518059 ,  0.04473177, -0.1945267 , ..., -2.335108  ,
         -2.4626963 , -5.0024285 ],
        [ 2.3648248 ,  0.97710556,  1.012191  , ..., -1.0877454 ,
         -0.6660234 , -3.102068  ]],

       [[ 3.7218885 , -0.84873706, -1.3898596 , ..., -4.8161006 ,
         -4.570894  , -8.429901  ],
        [ 2.07811   , -0.4502775 , -0.83742374, ..., -3.2340186 ,
         -3.0151343 , -5.6133137 ],
        [ 2.7128308 ,  0.83468133,  0.983893  , ..., -1.5471401 ,
         -1.1232706 , -4.1285787 ]],

       [[ 3.091966  , -0.7372267 , -1.5903194 , ..., -4.4885645 ,
         -4.027539  , -6.5031567 ],
        [ 3.1309593 , -0.27288455, -0.38468724, ..., -2.6189554 ,
         -2.7946172 , -5.7390475 ],
        [ 2.9332244 ,  0.91419905,  0.36722118, ..., -2.2392943 ,
         -1.711731  , -4.815379  ]],

       ...,

       [[ 3.608546  , -0.42307097, -0.39934373, ..., -4.085698 

In [92]:
prediction.shape

(738, 3, 30522)

In [93]:
predicted_indices = np.argmax(prediction, axis=-1)

In [94]:
predicted_indices

array([[ 40, 116,  54],
       [ 44, 116,  86],
       [ 56, 116,  75],
       ...,
       [ 60, 114,  50],
       [ 63, 116,  75],
       [ 16,  44,  45]], dtype=int64)

In [95]:
tokenizer.detokenize(predicted_indices)

<tf.Tensor: shape=(738,), dtype=string, numpy=
array([b'CAG GGM GAU', b'CUG GGM AMU', b'GAG GGM MUC', b'GGG GGU UUC',
       b'GGU GGM MUC', b'GAG GGM CGC', b'AUG AAM MUC', b'CCC GGM CGC',
       b'AAG CCC AMU', b'AAG GAA CCG', b'GGU GGM MUC', b'GGU GGM MUC',
       b'CCC GGM UUU', b'GUU GGM CCG', b'CCC GGM CCG', b'AAG AAC CUA',
       b'GAG GGM UUU', b'AUG AUM CUG', b'GAG GGM CGG', b'GGA GGM MUC',
       b'GUG GGM CCG', b'GUG GGM CCG', b'AAG GGM CCC', b'GGA GGM MUC',
       b'GGU GGM MUC', b'GGU GAU MUC', b'GGA GGM MUC', b'GAG GGU CCU',
       b'GGG GGM CCU', b'GGG GGM MUC', b'GGA GGM MUC', b'GCA GAA MUC',
       b'CUG GGG MUC', b'UAG GGM MUU', b'AUG GGM CUU', b'GGU GGM GMU',
       b'GGG GGU UUC', b'GUG GGM CCG', b'GGU GGM MUC', b'AAG AAM CUG',
       b'GUG GGM CCG', b'AAG GGM UMC', b'AAG GGM CCG', b'GGA GGM MUC',
       b'AUC UCC CCC', b'GAG GGM MUC', b'AUG AUM CUG', b'AAG GGM CCG',
       b'GUG GGM CCG', b'GGU GGM MUC', b'AAG GGM MUC', b'GCG GGM CCG',
       b'CAG GGM CUA', b'GUG G

In [96]:
label

array([['CAM', 'AMA', 'MAU'],
       ['UUM', 'UMU', 'MUU'],
       ['GAM', 'AMU', 'MUC'],
       ...,
       ['GUM', 'UMG', 'MGU'],
       ['GCM', 'CMU', 'MUC'],
       ['ACM', 'CMC', 'MCA']], dtype='<U32')

In [97]:
argmax_indices = np.argmax(prediction, axis=-1)

In [98]:
top_k_indices = np.argsort(prediction, axis=-1)[:, :, -5:]

In [99]:
top_k_indices

array([[[ 39,  37,   8,  13,  40],
        [103,  20, 109, 113, 116],
        [ 22,  70,   6,  38,  54]],

       [[ 26,  32,  29,  28,  44],
        [ 43, 102, 110,  27, 116],
        [ 26,  98,  42,  74,  86]],

       [[ 65,  55,  54,  53,  56],
        [114, 101,  55, 113, 116],
        [ 43,  86,  98,  27,  75]],

       ...,

       [[ 57,  59,  66,  58,  60],
        [110,  57, 113, 116, 114],
        [116,  86,  66,  18,  50]],

       [[ 68,  61,  62,  64,  63],
        [110, 115, 111,  46, 116],
        [ 86,  98,  43,  27,  75]],

       [[ 13,  14,   8,  15,  16],
        [113,  68,  60, 116,  44],
        [ 61,  13,  53,  29,  45]]], dtype=int64)

In [100]:
tokenizer.detokenize(top_k_indices)

<tf.RaggedTensor [[b'CAC CAA AAG ACA CAG', b'ACM AGG CAM GAM GGM', b'UAU MAU AAU CAU GAU'],
 [b'UUU UCG UCA UUG CUG', b'CUC AUM CUM UUC GGM', b'UUU GMU CUU MUU AMU'],
 [b'GGA GAC GAU GAA GAG', b'GUM AAM GAC GAM GGM', b'CUC AMU GMU UUC MUC'],
 ...,
 [b'GUA GUC GGU GUU GUG', b'CUM GUA GAM GGM GUM', b'GGM AMU GGU AGU CGU'],
 [b'GGG GCA GCU GCG GCC', b'CUM GCM CCM CCU GGM', b'AMU GMU CUC UUC MUC'],
 [b'ACA ACU AAG ACC ACG', b'GAM GGG GUG GGM CUG', b'GCA ACA GAA UCA CCA']]>